# 📐 Phase 3 — Feature Extraction

**Project:** Facial Skin Condition Detection  
**Level:** MCA Academic Project  

---

## Overview

This notebook demonstrates the feature extraction pipeline in `src/feature_extraction.py`.

### Feature Vector (58 dimensions total)

| Group | Dimensions | Description |
|-------|-----------|-------------|
| **Colour (RGB)** | 6 | Mean + Std for R, G, B channels |
| **Colour (HSV)** | 6 | Mean + Std for H, S, V channels |
| **GLCM Texture** | 20 | Contrast, Dissimilarity, Homogeneity, Energy, Correlation @ 0°, 45°, 90°, 135° |
| **LBP Texture** | 26 | Local Binary Pattern histogram (P=8, R=1, uniform) |
| **Total** | **58** | Combined feature vector |

> **Why no deep features?** This is a traditional ML project. Handcrafted features allow interpretability and require far fewer samples than CNN-based approaches.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import load_and_preprocess_image
from feature_extraction import (
    extract_color_features,
    extract_glcm_features,
    extract_lbp_features,
    extract_all_features,
    run_feature_extraction,
    _get_feature_names,
)

print('Imports OK')

## 1. Feature Names

In [ ]:
feature_names = _get_feature_names()
print(f'Total features: {len(feature_names)}\n')
for i, name in enumerate(feature_names, 1):
    print(f'  {i:2d}. {name}')

## 2. Visualise Individual Feature Groups on a Sample Image

In [ ]:
DATASET_DIR  = PROJECT_ROOT / 'dataset'
SUPPORTED_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}

# Find any sample image
sample_img_path = None
for sub in DATASET_DIR.iterdir():
    if sub.is_dir():
        imgs = [f for f in sub.rglob('*') if f.suffix.lower() in SUPPORTED_EXT]
        if imgs:
            sample_img_path = imgs[0]
            break

if sample_img_path is None:
    print('⚠ No sample image found — add images to dataset/ first.')
else:
    print(f'Using sample: {sample_img_path}')
    img_bgr = load_and_preprocess_image(sample_img_path)
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    color_feats = extract_color_features(img_bgr)
    glcm_feats  = extract_glcm_features(gray)
    lbp_feats   = extract_lbp_features(gray)
    all_feats   = extract_all_features(img_bgr)

    print(f'\n  colour features : {color_feats.shape}  → {color_feats.round(2)}')
    print(f'  GLCM features   : {glcm_feats.shape}  → {glcm_feats.round(4)}')
    print(f'  LBP features    : {lbp_feats.shape}   → {lbp_feats.round(4)}')
    print(f'  Combined vector : {all_feats.shape}')

In [ ]:
if sample_img_path:
    from skimage.feature import local_binary_pattern

    lbp_map = local_binary_pattern(gray, P=8, R=1, method='uniform')

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle('Feature Visualisation on Sample Image', fontsize=13, fontweight='bold')

    axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Preprocessed Image'); axes[0].axis('off')

    axes[1].imshow(gray, cmap='gray')
    axes[1].set_title('Grayscale'); axes[1].axis('off')

    axes[2].imshow(lbp_map, cmap='viridis')
    axes[2].set_title('LBP Map (P=8, R=1)'); axes[2].axis('off')

    axes[3].bar(range(len(lbp_feats)), lbp_feats, color='steelblue', edgecolor='white', linewidth=0.3)
    axes[3].set_title('LBP Histogram (26 bins)')
    axes[3].set_xlabel('Bin'); axes[3].set_ylabel('Normalised frequency')

    plt.tight_layout()
    plt.show()

## 3. Run Full Feature Extraction

This cell calls `run_feature_extraction()` which processes **all** images in `dataset/` and writes `features/features.csv`.

In [ ]:
df = run_feature_extraction()
df.head()

## 4. Feature Distribution Analysis

In [ ]:
if 'df' in dir() and df is not None and not df.empty:
    print(f'Dataset shape: {df.shape}')
    print('\nClass distribution:')
    print(df['label'].value_counts())
    print('\nFeature statistics:')
    feat_cols = [c for c in df.columns if c not in ('label', 'filepath')]
    df[feat_cols].describe().round(4)

In [ ]:
if 'df' in dir() and df is not None and not df.empty:
    feat_cols = [c for c in df.columns if c not in ('label', 'filepath')]
    # Plot first 12 colour features as boxplot by class
    color_cols = feat_cols[:12]
    fig, axes = plt.subplots(3, 4, figsize=(18, 10))
    axes = axes.flatten()
    fig.suptitle('Colour Feature Distributions by Class', fontsize=14, fontweight='bold')
    palette = {'Acne': '#E74C3C', 'Rosacea': '#E67E22', 'Melasma': '#9B59B6',
               'Facial Eczema': '#F1C40F', 'Seborrheic Dermatitis': '#2ECC71'}
    for i, col in enumerate(color_cols):
        sns.boxplot(data=df, x='label', y=col, ax=axes[i], palette=palette)
        axes[i].set_title(col, fontsize=9)
        axes[i].set_xlabel('')
        axes[i].tick_params(axis='x', rotation=30, labelsize=7)
    plt.tight_layout()
    (PROJECT_ROOT / 'reports').mkdir(exist_ok=True)
    plt.savefig(PROJECT_ROOT / 'reports' / 'feature_distributions.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved → reports/feature_distributions.png')

## 5. Feature Correlation Heatmap

In [ ]:
if 'df' in dir() and df is not None and not df.empty:
    feat_cols = [c for c in df.columns if c not in ('label', 'filepath')]
    corr = df[feat_cols].corr()
    fig, ax = plt.subplots(figsize=(16, 12))
    sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0,
                xticklabels=False, yticklabels=False,
                linewidths=0, square=True, vmin=-1, vmax=1)
    ax.set_title('Feature Correlation Matrix (58 × 58)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'reports' / 'feature_correlation.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved → reports/feature_correlation.png')

---
## Summary

- **58 features** extracted per image: 12 colour + 20 GLCM + 26 LBP
- Saved to `features/features.csv`

**Next:** Run `04_model_training.ipynb` to train the 7 ML models.